[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/v04_section.ipynb)

# Where does a seesaw balance? The section formula and the centroid

A 30 kg child and a 60 kg adult share a 3 m seesaw whose pivot can slide. The pivot goes 2 m from the child: the
balance point is a weighted average of where the two sit. Written with a ratio, the weights cross over, and that is
the section formula. The same move, done twice, finds the balance point of a triangle.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
▶ Run button) to run it. Then run each cell in turn, from top to bottom. When sliders appear, drag them. Change any
number you like and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in these notebooks. Run this cell first.
import math, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from ipywidgets import interact, FloatSlider, IntSlider

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#8fcfc4", "#a9dca6", "#c3a9e6"   # the "along" part · the result · the "across" part

warnings.filterwarnings("ignore", message=".*as float was deprecated.*")   # a browser-only notice, harmless
plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def fmt(v):
    """A number as the board writes it: no trailing zeros, a true minus sign."""
    return f"{v:.2f}".rstrip("0").rstrip(".").replace("-", "−")

def column(ax, vals, x, y, size=24, col=CHALK, name="", gap=0.14):
    """An arrow's numbers as a column in square brackets on a flat panel (top row at y)."""
    for k, v in enumerate(vals):
        ax.text(x, y - k * gap, fmt(v), color=col, fontsize=size, ha="center", va="center", family="serif")
    top, bot, w = y + gap * 0.55, y - (len(vals) - 1) * gap - gap * 0.55, 0.12
    for side in (-1, 1):
        bx = x + side * w
        ax.plot([bx, bx], [bot, top], color=DIM, lw=2)
        for yy in (top, bot): ax.plot([bx, bx - side * 0.03], [yy, yy], color=DIM, lw=2)
    if name: ax.text(x - w - 0.04, y - gap * (len(vals) - 1) / 2, name, color=col, fontsize=size, ha="right",
                     va="center", family="serif")

**The drawing tools.** Run the next cell too: it holds the code that draws each figure below. You don't need to read it
to play along.

In [ ]:
def seesaw(child=30, adult=60):
    L = 3.0
    p = (child * 0 + adult * L) / (child + adult)
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.set_xlim(-0.6, 3.6); ax.set_ylim(-0.95, 1.55); ax.set_aspect("equal"); ax.axis("off")
    ax.plot([-0.5, 3.5], [-0.62, -0.62], color=DIM, lw=1.5)                     # the ground
    ax.add_patch(Polygon([[p, 0], [p - 0.22, -0.62], [p + 0.22, -0.62]], color=CORAL))   # the pivot
    ax.add_patch(Rectangle((0, 0), L, 0.07, color=CHALK))                       # the plank
    for x, w, who in ((0, child, "child"), (L, adult, "adult")):
        s = 0.18 + 0.32 * math.sqrt(w / 60)                                    # a block, its area ∝ weight
        ax.add_patch(Rectangle((x - s / 2, 0.07), s, s, facecolor=SLATE2, edgecolor=CHALK, lw=2))
        ax.text(x, 0.07 + s + 0.12, f"{who}\n{w} kg", ha="center", va="bottom", fontsize=13, color=CHALK)
    for k in range(4):                                                         # a ruler along the ground
        ax.plot([k, k], [-0.62, -0.7], color=DIM, lw=1.5); ax.text(k, -0.85, f"{k} m", ha="center", color=DIM, fontsize=11)
    ax.annotate("", (p, 0.92), (0, 0.92), arrowprops=dict(arrowstyle="<->", color=TEAL, lw=1.8))
    ax.text(p / 2, 0.97, f"{fmt(p)} m", ha="center", color=TEAL, fontsize=13)
    ax.set_title(f"pivot {fmt(p)} m from the child, {fmt(L - p)} m from the adult:   "
                 f"{child} × {fmt(p)} = {fmt(child * p)}   and   {adult} × {fmt(L - p)} = {fmt(adult * (L - p))}",
                 color=CORAL, fontsize=13)
    plt.show()

A, B = np.array([1.0, 1.0]), np.array([2.8, 3.4])

def section(k=0.667):
    r = A + k * (B - A)                                    # k of the way from A to B
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.05, 0.1, 0.55, 0.85])
    ax.set_xlim(-0.8, 6.2); ax.set_ylim(-0.6, 6.4); ax.set_aspect("equal")
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    far = [A + t * (B - A) for t in (-1.0, 2.4)]
    ax.plot([far[0][0], far[1][0]], [far[0][1], far[1][1]], "--", color=GHOST, lw=1.5)   # the line, both ways
    ax.plot([A[0], B[0]], [A[1], B[1]], color=CHALK, lw=5, solid_capstyle="round")      # the plank
    ax.annotate("", r, (0, 0), arrowprops=dict(arrowstyle="-|>", color=SAGE, lw=2.6, mutation_scale=22))
    for q, lab in ((A, "A (child)"), (B, "B (adult)")):
        ax.plot(*q, "o", color=CHALK, ms=9); ax.text(q[0] + 0.15, q[1] - 0.3, lab, color=CHALK, fontsize=12)
    ax.plot(*r, "o", color=CORAL, ms=12)
    ax.plot(0, 0, "s", color=DIM, ms=8); ax.text(0.12, -0.35, "gate O", color=DIM, fontsize=11)
    p = fig.add_axes([0.62, 0.0, 0.38, 1.0]); p.axis("off"); p.set_xlim(0, 1); p.set_ylim(0, 1)
    column(p, r, 0.55, 0.8, col=SAGE, name="r =")
    da, db = np.linalg.norm(r - A), np.linalg.norm(r - B)
    inside = 0 <= k <= 1
    p.text(0.5, 0.5, f"{fmt(da)} m from A,  {fmt(db)} m from B", ha="center", color=CHALK, fontsize=13)
    if db > 1e-9 and da > 1e-9:
        p.text(0.5, 0.38, f"ratio  {fmt(da / db)} : 1   ({'inside' if inside else 'outside'})", ha="center",
               color=CORAL, fontsize=16)
    p.text(0.5, 0.24, f"r = a + {fmt(k)}(b − a)", ha="center", color=TEAL, fontsize=14)
    p.text(0.5, 0.14, f"  = {fmt(1 - k)} a + {fmt(k)} b", ha="center", color=TEAL, fontsize=14)
    plt.show()

def centroid(cx=3.0, cy=6.0):
    P = np.array([[0.0, 0.0], [6.0, 0.0], [cx, cy]])
    g = P.mean(axis=0)
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.05, 0.1, 0.55, 0.85])
    ax.set_xlim(-1, 9); ax.set_ylim(-1.5, 8); ax.set_aspect("equal")
    ax.add_patch(Polygon(P, facecolor=SLATE2, edgecolor=CHALK, lw=2.5))
    for i in range(3):                                     # each corner to the middle of the opposite side
        mid = (P[(i + 1) % 3] + P[(i + 2) % 3]) / 2
        ax.plot([P[i][0], mid[0]], [P[i][1], mid[1]], color=SAGE, lw=1.6)
        ax.plot(*mid, "o", color=SAGE, ms=5)
    for q, lab in zip(P, "abc"): ax.text(q[0], q[1] + 0.35 if lab == "c" else q[1] - 0.7, lab, color=CHALK,
                                         fontsize=15, ha="center", family="serif")
    ax.plot(*g, "o", color=CORAL, ms=13)
    ax.plot([g[0], g[0]], [g[1] - 0.15, -1.4], color=DIM, lw=3)        # the pencil underneath
    ax.set_xlabel("x"); ax.set_ylabel("y")
    p = fig.add_axes([0.62, 0.0, 0.38, 1.0]); p.axis("off"); p.set_xlim(0, 1); p.set_ylim(0, 1)
    p.text(0.5, 0.88, "centroid = (a + b + c) ÷ 3", ha="center", color=CHALK, fontsize=15)
    column(p, g, 0.55, 0.66, col=CORAL, name="g =")
    p.text(0.5, 0.36, f"x: (0 + 6 + {fmt(cx)}) ÷ 3 = {fmt(g[0])}", ha="center", color=DIM, fontsize=13)
    p.text(0.5, 0.27, f"y: (0 + 0 + {fmt(cy)}) ÷ 3 = {fmt(g[1])}", ha="center", color=DIM, fontsize=13)
    p.text(0.5, 0.14, "the three medians meet here", ha="center", color=SAGE, fontsize=13)
    plt.show()

## 1 · The seesaw

A light plank 3 m long, the child on the left end and the adult on the right. Where must the pivot go so the plank
balances? Each weight times its distance from the pivot must come out the same on both sides. Measured from the
child, that puts the pivot at the **weighted average** of the two seats:

pivot = (child × 0 + adult × 3) ÷ (child + adult)

Drag the adult's weight up and down and watch the pivot slide to keep the balance.

In [ ]:
interact(seesaw, child=IntSlider(value=30, min=10, max=90, step=5, description="child (kg)"),
         adult=IntSlider(value=60, min=10, max=120, step=5, description="adult (kg)"));

**Notice.** Twice the weight sits at half the distance. The pivot is pulled towards the heavier person: with
30 kg and 60 kg it sits two thirds of the way along, and two thirds is the adult's share of the weight, 60 out of 90.

## 2 · A point that cuts a line in a ratio

In a playground the seesaw can point any way, so each seat needs two numbers: the child's seat **a** = (1, 1), the
adult's **b** = (2.8, 3.4), measured from the gate O. The point that cuts AB in the ratio **m : n** is

r = (n a + m b) ÷ (m + n)

The weights cross over: the m, the piece next to A, multiplies b. Drag the slider to move the point along the line:
inside the plank (between A and B) and outside it, past either end. The arrow from the gate and its column of
numbers always agree.

In [ ]:
interact(section, k=FloatSlider(value=0.667, min=-0.5, max=2.0, step=1 / 12, readout_format=".3f",
                                description="along AB"));

**Notice.** At two thirds of the way (k = 0.667), the point is the pivot (2.2, 2.6), and it cuts AB 2 : 1, yet the
formula gives a the 1/3 and b the 2/3. At k = 2 it is outside, past B, twice as far from A as from B: the same
ratio 2 : 1, found with a minus sign, (m b − n a) ÷ (m − n).

In [ ]:
def divide(a, b, m, n, outside=False):
    """The point dividing AB in the ratio m : n, inside (or outside), from the formula."""
    a, b = np.array(a, float), np.array(b, float)
    r = (m * b - n * a) / (m - n) if outside else (n * a + m * b) / (m + n)
    print(f"{'outside' if outside else 'inside'} {m} : {n} →", tuple(fmt(v) for v in r))
    return r

divide((-1, 2, 3), (5, -1, 0), 2, 1)                 # the textbook question: inside …
divide((-1, 2, 3), (5, -1, 0), 2, 1, outside=True)   # … and outside

## 3 · A triangle balanced on its centroid

Archimedes showed that a triangle balances where the lines from each corner to the middle of the opposite side (the
**medians**) meet. That point, the **centroid**, cuts each median 2 : 1 from the corner, so it is
(a + b + c) ÷ 3, the average of the three corners. Move the top corner with the sliders.

In [ ]:
interact(centroid, cx=FloatSlider(value=3.0, min=-1, max=8, step=0.5, description="c: x"),
         cy=FloatSlider(value=6.0, min=1, max=7.5, step=0.5, description="c: y"));

**Try it.** Put a second 30 kg child in the middle of the plank, at 1.5 m: where does the pivot go now? It is a
weighted average of three positions. (That is the challenge from the video.)

## Sources
Archimedes, *On the Equilibrium of Planes*, Book I, Propositions 6–7 and 13–14, in T. L. Heath, *The Works of
Archimedes* (Cambridge, 1897). NCERT Class XII Mathematics ch. 10 (Vector Algebra) and Class XI ch. 11; CBSE
Mathematics syllabus 2026–27. Retrieved 2026-10-26. The plank is light: its own weight is ignored.